# MultiHb — experiments for the revision

One notebook, run top to bottom. It produces every number and figure the revised paper
needs, and it reads the locked test set only for a list of configurations fixed in
advance.

| Block | What it does | Reads the test set |
|---|---|---|
| Stage 1 | ranks the 14 backbones on **validation** MAE | no |
| Stage 2 | ranks the 4 fusion operators on validation | no |
| Stage 3 | ranks the variable subsets on validation | no |
| Final | selected model and its image-only counterpart, several seeds | yes, pre-specified |
| Controls | shuffled-tabular capacity control; image + height, weight, BMI, age, sex | yes, pre-specified |
| Baselines | mean and median predictors; tabular-only regressors | yes, no image model |
| Analysis | paired bootstrap, confidence intervals, R2, Bland-Altman, error by Hb and age band | reads saved CSVs |
| Out of range | scores the trained model on Hb values outside the inclusion range | separate rows |
| Device | CPU latency, memory and agreement with the saved GPU predictions | reads saved CSVs |

Set `RUN_PROFILE = "smoke"` first. It trains 2 epochs on 2 folds and proves the whole
pipeline end to end in minutes. Then switch to `"full"`.

`RESUME = True` means an interrupted run restarts where it stopped.


## Section 1 — Configuration (edit only this block)

In [ ]:
# ==============================================================
#            ALL VARIABLES -- EDIT ONLY THIS BLOCK
# ==============================================================

# ---- Cross-validation --------------------------------------------------------
K = 5                       # folds drawn from the train pool (K>=2)
assert K >= 2, "nested selection needs K>=2"

# ---- Task --------------------------------------------------------------------
TASK = "regression"

# ---- Dataset paths -----------------------------------------------------------
IMAGE_DIR   = "/media/junaid/DATA/JUNAID/ANEMIA/dataset/left_eye"
CSV_PATH    = "/media/junaid/DATA/JUNAID/ANEMIA/dataset/merge_excel_1.xlsx"
IMAGE_DIR_2 = ""            # second image slot unused: one image per subject
CSV_PATH_2  = ""

# ---- Column names (must match the sheet headers EXACTLY) ---------------------
IMAGE_COL  = "Patient ID"
HB_COL     = "Haemoglobin (gm/dL)"
HEIGHT_COL = "Height"
WEIGHT_COL = "Weight"
BMI_COL    = "BMI"                 # derived below, not read from the sheet
DOB_COL    = "Date Of Birth"       # used for age in months
AGE_COL    = "_age_months"         # derived below
SEX_NUM_COL = "_sex01"             # derived below: 1 = male, 0 = female
VISIT_COL  = "Created On"          # used for age in months
SEX_COL    = "Gender"

# ---- Tabular variables -------------------------------------------------------
USE_TABULAR      = True
NUMERIC_FEATURES = [HEIGHT_COL, WEIGHT_COL, BMI_COL]

FUSION       = "attn"                                 # default for the main pass
FUSION_MODES = ["concat", "film", "gate", "attn"]     # Stage 2 sweep
TAB_HIDDEN   = 64
FUSE_DIM     = 128
TAB_DROPOUT  = 0.2

# ---- Hb range ----------------------------------------------------------------
# Lower bound keeps genuine severe anemia (WHO severe is <7.0 g/dL); the cohort has a
# dense, continuous tail down to ~4 g/dL. Upper bound drops entries that are not
# physiological for this age range (9 values above 17, up to 23.4) and read as
# transcription error.
HB_FILTER_MIN = 4.0
HB_FILTER_MAX = 17.0
USE_AUGMENTATION = True
AUG_BINS         = 10

# ---- Training ----------------------------------------------------------------
EPOCHS      = 40
BATCH_SIZE  = 32
LR          = 1e-4
TEST_SPLIT  = 0.1           # locked, scored once
NUM_WORKERS = 4
SEED        = 42
EARLY_STOP  = 8
SCHEDULER   = "cosine"
GRAD_ACCUM  = 1
WARMUP_EPOCHS = 3
USE_AMP     = True
FREEZE_BACKBONE = True
IMG_SIZE    = 224

PREPROCESS = {"colorspace": "RGB", "use_clahe": False,
              "clahe_clip": 2.0, "clahe_grid": (8, 8)}

# ---- Regression --------------------------------------------------------------
# Target scaling uses the same interval as the inclusion filter, so the two cannot drift.
REG_CONFIG = {"normalize_hb": True,
              "hb_min": HB_FILTER_MIN, "hb_max": HB_FILTER_MAX,
              "loss_fn": "huber", "huber_delta": 1.0}

NEW_MODEL_CFG = {"embed_dim": 128, "depth": 4, "d_state": 64,
                 "colour_proj": "learned", "n_experts": 5}

# ---- The fourteen backbones reported in the paper ----------------------------
RUN = {
    # 9 Mamba variants
    "Mamba1": True, "Mamba2": True, "Mamba3": True, "VMamba": True,
    "MambaVision": True, "MedMamba": True,
    "EfficientMamba": True, "AdaptiveScan": True, "CrossFusion": True,
    # 5 pretrained CNN / ViT baselines
    "ResNet50": True, "EfficientNetB3": True, "DenseNet121": True,
    "MobileNetV3": True, "ViT_B16": True,
}

# ---- Stage control -----------------------------------------------------------
RUN_STAGE1 = True    # backbone selection      (validation only)
RUN_STAGE2 = True    # fusion selection        (validation only)
RUN_STAGE3 = True    # variable selection      (validation only)
RUN_FINAL  = True    # single locked-test evaluation of the pre-specified configs
RUN_BASELINES    = True    # mean/median predictors and tabular-only regressors
RUN_CONTROLS     = True    # shuffled-tabular capacity control; image + all five variables
RUN_OUT_OF_RANGE = True    # score the trained model on Hb outside the inclusion range
RUN_DEVICE_BENCHMARK = False   # CPU latency / memory / agreement; set True on the handheld

# Stage 1 compares backbones on the image alone, as the paper states. Set this to
# NUMERIC_FEATURES only if the paper is changed to say Stage 1 ran with fusion.
STAGE1_FEATS = []

# Seeds for the FINAL configurations. Folds and the test split stay fixed (SEED above);
# these change weight initialisation, augmentation and sampling order.
FINAL_SEEDS = [42, 43, 44]

SAVE_CHECKPOINTS = True    # keep the fold models of the selected configuration
N_BOOT = 10000             # bootstrap resamples for the confidence intervals

# ---- Choices the Methods section must state exactly as set here ---------------
FREEZE_TIMM      = False   # True trains only the head of the pretrained CNN/ViT baselines
AUG_COLOR_JITTER = True    # colour jitter in training augmentation

# Stage 3 variable subsets, evaluated on validation
VARIABLE_STEPS = [
    ("image-only",        []),
    ("+Height",           [HEIGHT_COL]),
    ("+Weight",           [WEIGHT_COL]),
    ("+BMI",              [BMI_COL]),
    ("+Height+Weight+BMI",[HEIGHT_COL, WEIGHT_COL, BMI_COL]),
]

# ---- WHO age-specific anemia thresholds (2011), used for stratification and for
#      the screening metrics computed after the final evaluation ---------------
def who_threshold(age_months, sex=None):
    """Hb (g/dL) below which anemia is defined, WHO 2011."""
    if age_months is None or age_months != age_months:   # NaN
        return 11.0
    if age_months < 60:     return 11.0      # 6-59 months (also applied below 6 months)
    if age_months < 144:    return 11.5      # 5-11 years
    if age_months < 180:    return 12.0      # 12-14 years
    return 13.0 if str(sex).upper().startswith("M") else 12.0

# ---- Run profile -------------------------------------------------------------
# "smoke" trains 2 epochs on 2 folds so the whole pipeline can be verified in
# minutes before committing to the full run. Switch to "full" for the real run.
RUN_PROFILE = "full"            # "smoke" | "full"
if RUN_PROFILE == "smoke":
    EPOCHS, K = 2, 2
    FINAL_SEEDS = FINAL_SEEDS[:1]
    N_BOOT = 500
    RUN = {k: (k in ("ViT_B16", "MobileNetV3")) for k in RUN}   # two backbones only
    print("[smoke] EPOCHS=2, K=2 -- results are NOT publishable, pipeline check only")

RESUME = True                   # skip configurations already present in the results CSV

# ---- Output files ------------------------------------------------------------
OUT_PREFIX = "multihb_exp"

# ---- Repo path autodetect ----------------------------------------------------
import os, sys
def _find_repo_root():
    cands = [os.getcwd(), os.path.join(os.getcwd(), "MULTIMAMBA")]
    try:
        for d in os.listdir(os.getcwd()):
            p = os.path.join(os.getcwd(), d)
            if os.path.isdir(p): cands.append(p)
    except Exception: pass
    for c in cands:
        if os.path.isdir(os.path.join(c, "MAMBA_MODELS")): return c
    return os.getcwd()

REPO_ROOT  = _find_repo_root()
MODELS_DIR = os.path.join(REPO_ROOT, "MAMBA_MODELS")
if MODELS_DIR not in sys.path: sys.path.insert(0, MODELS_DIR)

print(f"TASK        : {TASK}  (regression only)")
print(f"Cross-val   : K={K} folds on the train pool, test set scored once")
print(f"Tabular     : {NUMERIC_FEATURES}")
print(f"Hb range    : [{HB_FILTER_MIN}, {HB_FILTER_MAX}] g/dL, filter and target scaling")
print(f"MODELS_DIR  : {MODELS_DIR}")
print(f"Backbones ON: {len([k for k,v in RUN.items() if v])}")
print(f"Final seeds : {FINAL_SEEDS}   timm frozen: {FREEZE_TIMM}   colour jitter: {AUG_COLOR_JITTER}")

## Section 2 — Install dependencies (Kaggle / fresh environment)

In [ ]:
# ---- Uncomment on a fresh environment ----
# import subprocess, sys
# subprocess.run([sys.executable, "-m", "pip", "install", "-q",
#                 "timm", "einops", "pandas", "openpyxl", "scikit-learn", "tqdm",
#                 "matplotlib", "scipy"], check=False)
print("dependency cell skipped (uncomment if needed)")

## Section 3 — Imports and device

In [ ]:
import os, sys, math, time, json, glob, random, warnings, traceback, importlib.util, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import r2_score

warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def _fn(name):
    """Config name -> safe file-name fragment."""
    return "".join(ch if ch.isalnum() or ch in "-_+" else "_" for ch in str(name))

seed_all(SEED)
print("device:", DEVICE, "| torch", torch.__version__)

## Section 4 — Data: merge, derive BMI and age, lock the test set

Each subject contributes exactly one left-eye image, so a row is a subject and splitting
rows is splitting subjects. The split is stratified on the **age-specific** WHO anemia
label rather than a single 12.0 g/dL cut.

In [ ]:

from common.augment import merge_and_filter_datasets, make_balanced_loader

def build_transforms(augment):
    steps = []
    if augment:
        steps += [transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
                  transforms.RandomCrop(IMG_SIZE),
                  transforms.RandomHorizontalFlip(),
                  transforms.RandomRotation(15)]
        if AUG_COLOR_JITTER:
            steps.append(transforms.ColorJitter(0.25, 0.25, 0.15))
    else:
        steps.append(transforms.Resize((IMG_SIZE, IMG_SIZE)))
    # Every model receives inputs scaled to [-1, 1]. The pretrained timm baselines
    # re-normalise to their own statistics inside TimmWrapper.
    steps += [transforms.ToTensor(), transforms.Normalize((0.5,)*3, (0.5,)*3)]
    return transforms.Compose(steps)

_IMG_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ""]
def find_image_path(pid, img_dir):
    for ext in _IMG_EXTS:
        p = os.path.join(img_dir, str(pid) + ext)
        if os.path.exists(p): return p
    return None

def _load_df(hb_lo, hb_hi, verbose=True):
    slot_dfs, slot_dirs = [], []
    for csv_p, img_d in [(CSV_PATH, IMAGE_DIR), (CSV_PATH_2, IMAGE_DIR_2)]:
        if csv_p and img_d:
            _df = (pd.read_excel(csv_p) if csv_p.endswith((".xlsx", ".xls"))
                   else pd.read_csv(csv_p))
            slot_dfs.append(_df); slot_dirs.append(img_d)
    if not slot_dfs:
        raise RuntimeError("No dataset slots active. Set CSV_PATH and IMAGE_DIR.")
    df = merge_and_filter_datasets(
        dataframes=slot_dfs, image_dirs=slot_dirs, hb_col=HB_COL, image_col=IMAGE_COL,
        hb_filter_min=hb_lo, hb_filter_max=hb_hi,
        find_image_path_fn=find_image_path, verbose=verbose)
    return df.reset_index(drop=True)

# ---- plausibility limits (a handful of records carry data-entry errors) ----
HEIGHT_RANGE_CM = (45.0, 200.0)
WEIGHT_RANGE_KG = (2.0, 120.0)
BMI_RANGE       = (5.0, 45.0)
AGE_RANGE_MO    = (0.0, 216.0)

def _clean(series, lo, hi, label, verbose=True):
    v = pd.to_numeric(series, errors="coerce")
    bad = v.notna() & ((v < lo) | (v > hi))
    if bad.sum() and verbose:
        print(f"  [clean] {label}: {int(bad.sum())} value(s) outside [{lo}, {hi}] set to NaN")
    v[bad] = np.nan
    return v

def _parse_visit(s):
    return pd.to_datetime(pd.Series(s).astype(str).str.slice(0, 24),
                          format="%a %b %d %Y %H:%M:%S", errors="coerce")

def derive_columns(df, verbose=True):
    """Sanitise height and weight, derive BMI, age in months, numeric sex and the
    age-specific WHO label. Used for the study rows and for the out-of-range rows."""
    df = df.copy()
    h = _clean(df.get(HEIGHT_COL), *HEIGHT_RANGE_CM, HEIGHT_COL, verbose)
    w = _clean(df.get(WEIGHT_COL), *WEIGHT_RANGE_KG, WEIGHT_COL, verbose)
    df[HEIGHT_COL], df[WEIGHT_COL] = h, w
    bmi = w / ((h / 100.0) ** 2)
    bmi[~np.isfinite(bmi)] = np.nan
    df[BMI_COL] = _clean(bmi, *BMI_RANGE, BMI_COL, verbose)
    if DOB_COL in df.columns and VISIT_COL in df.columns:
        dob   = pd.to_datetime(df[DOB_COL], errors="coerce")
        visit = _parse_visit(df[VISIT_COL]).values
        df[AGE_COL] = _clean((pd.Series(visit, index=df.index) - dob).dt.days / 30.4375,
                             *AGE_RANGE_MO, "age (months)", verbose)
    else:
        if verbose: print(f"  [WARN] {DOB_COL!r} or {VISIT_COL!r} missing -> age unavailable")
        df[AGE_COL] = np.nan
    sx = df.get(SEX_COL, pd.Series([None] * len(df), index=df.index)).astype(str).str.upper().str[:1]
    df[SEX_NUM_COL] = np.where(sx == "M", 1.0, np.where(sx == "F", 0.0, np.nan))
    df["_who_thr"] = [who_threshold(a, s) for a, s in zip(df[AGE_COL], sx)]
    df["_anemic"]  = (df[HB_COL] < df["_who_thr"]).astype(int)
    return df

# ---- everything with an image and a numeric Hb, before the inclusion filter ----
ALL_DF = derive_columns(_load_df(-1e9, 1e9, verbose=True), verbose=True)
_n_lo = int((ALL_DF[HB_COL] < HB_FILTER_MIN).sum())
_n_hi = int((ALL_DF[HB_COL] > HB_FILTER_MAX).sum())
OUT_DF = ALL_DF[(ALL_DF[HB_COL] < HB_FILTER_MIN) | (ALL_DF[HB_COL] > HB_FILTER_MAX)].reset_index(drop=True)
FULL   = ALL_DF[(ALL_DF[HB_COL] >= HB_FILTER_MIN) & (ALL_DF[HB_COL] <= HB_FILTER_MAX)].reset_index(drop=True)
if len(FULL) == 0:
    raise RuntimeError("Zero valid samples after filtering.")

print("\nCOHORT FLOW (report these numbers in the paper)")
print(f"  rows with an image and a numeric Hb : {len(ALL_DF)}")
print(f"  excluded, Hb < {HB_FILTER_MIN:<5}              : {_n_lo}")
print(f"  excluded, Hb > {HB_FILTER_MAX:<5}              : {_n_hi}")
print(f"  study set                           : {len(FULL)}")

print("\nColumn coverage:")
for c in NUMERIC_FEATURES + [AGE_COL, SEX_NUM_COL]:
    n_ok = pd.to_numeric(FULL[c], errors="coerce").notna().sum()
    print(f"  {c:>12}: {n_ok}/{len(FULL)} usable")

# ---- locked test split, stratified on the age-specific label ----
POOL_DF, TEST_DF = train_test_split(FULL, test_size=TEST_SPLIT,
                                    random_state=SEED, stratify=FULL["_anemic"])
POOL_DF = POOL_DF.reset_index(drop=True)
TEST_DF = TEST_DF.reset_index(drop=True)
POOL_LBL = POOL_DF["_anemic"].values

print(f"\n  Samples {len(FULL)}  |  below the age-specific WHO cut-off: "
      f"{int(FULL['_anemic'].sum())} ({FULL['_anemic'].mean()*100:.1f}%)")
print(f"  Train+Val pool {len(POOL_DF)}   Locked test {len(TEST_DF)}")
print(f"  Hb study set : mean {FULL[HB_COL].mean():.2f}  SD {FULL[HB_COL].std():.2f}  "
      f"min {FULL[HB_COL].min():.2f}  max {FULL[HB_COL].max():.2f}")
print(f"  Hb test set  : mean {TEST_DF[HB_COL].mean():.2f}  SD {TEST_DF[HB_COL].std():.2f}")
print(f"  sex          : {FULL.get(SEX_COL).value_counts().to_dict() if SEX_COL in FULL else 'n/a'}")
print(f"  age (months) : median {FULL[AGE_COL].median():.0f}  "
      f"IQR {FULL[AGE_COL].quantile(.25):.0f}-{FULL[AGE_COL].quantile(.75):.0f}")


## Section 5 — Tabular preprocessing and the multimodal dataset

In [ ]:

def fit_tab_stats(df, numeric_feats):
    """Per-fold standardisation stats from TRAIN rows only."""
    stats = {}
    for c in numeric_feats:
        v = pd.to_numeric(df[c], errors="coerce") if c in df.columns else pd.Series(dtype=float)
        m = float(v.mean()) if v.notna().any() else 0.0
        s = float(v.std())  if v.notna().any() else 1.0
        stats[c] = (m, max(s, 1e-6))
    return stats

def _row_tab_vector(row, numeric_feats, stats):
    vec = []
    for c in numeric_feats:
        m, s = stats[c]
        x = pd.to_numeric(pd.Series([row.get(c, np.nan)]), errors="coerce").iloc[0]
        vec.append(0.0 if pd.isna(x) else (float(x) - m) / s)   # missing -> fold mean
    return np.asarray(vec, dtype=np.float32)

class MMDataset(Dataset):
    """Returns (image, tabular, hb, row_index).

    shuffle_tab=True pairs every image with the tabular vector of a different, fixed
    subject. The model keeps its full capacity but the variables carry no information
    about the subject in the image: the capacity control."""
    def __init__(self, df, transform, numeric_feats, stats, shuffle_tab=False, perm_seed=0):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.numeric_feats = numeric_feats
        self.stats = stats
        self.tab_dim = len(numeric_feats)
        self.perm = (np.random.RandomState(perm_seed).permutation(len(self.df))
                     if shuffle_tab else None)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        hb  = float(row[HB_COL])
        img = Image.open(row["_img_path"]).convert("RGB")
        if self.transform: img = self.transform(img)
        trow = row if self.perm is None else self.df.iloc[int(self.perm[idx])]
        tab = torch.from_numpy(_row_tab_vector(trow, self.numeric_feats, self.stats)) \
              if self.tab_dim else torch.zeros(0, dtype=torch.float32)
        return img, tab, torch.tensor(hb, dtype=torch.float32), torch.tensor(idx)

print("Tabular + MMDataset ready.")


## Section 6 — Fusion wrapper

The pre-hook is registered on the backbone's **`reg_head`** when it exists, which is the
head this study trains and reports, and falls back to `cls_head` for older builders.

In [ ]:
def _get_head(image_model):
    """The terminal head the pre-hook attaches to. reg_head is the reported head."""
    for attr in ("reg_head", "cls_head"):
        h = getattr(image_model, attr, None)
        if h is not None:
            return attr, h
    raise AttributeError("backbone exposes neither reg_head nor cls_head")

def _infer_feat_dim(head):
    if isinstance(head, nn.Linear): return head.in_features
    for m in head.modules():
        if isinstance(m, nn.Linear):   return m.in_features
        if isinstance(m, nn.LayerNorm):
            ns = m.normalized_shape
            return ns[0] if isinstance(ns, (tuple, list)) else int(ns)
    raise ValueError("Could not infer feat_dim")

def _mlp(inp, hid):
    return nn.Sequential(nn.Linear(inp, hid), nn.GELU(),
                         nn.LayerNorm(hid), nn.Linear(hid, hid), nn.GELU())

def _make_reg_head(dim):
    return nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, max(dim // 2, 8)),
                         nn.GELU(), nn.Linear(max(dim // 2, 8), 1))

class MultiModalWrapper(nn.Module):
    """Captures the pooled image feature with a forward pre-hook and fuses it with
    the encoded tabular vector. The backbone is never edited."""
    def __init__(self, image_model, tab_dim, fusion="attn",
                 tab_hidden=None, fuse_dim=None, dropout=None):
        super().__init__()
        tab_hidden = TAB_HIDDEN if tab_hidden is None else tab_hidden
        fuse_dim   = FUSE_DIM   if fuse_dim   is None else fuse_dim
        dropout    = TAB_DROPOUT if dropout   is None else dropout
        assert tab_dim > 0, "use ImageOnlyWrapper when tab_dim == 0"
        self.image_model = image_model
        self.fusion, self.tab_dim = fusion, tab_dim
        self.head_name, head = _get_head(image_model)
        feat_dim = _infer_feat_dim(head)
        self.feat_dim, self._feat = feat_dim, None
        head.register_forward_pre_hook(self._hook)

        if fusion == "concat":
            self.tab_enc = _mlp(tab_dim, tab_hidden); fused = feat_dim + tab_hidden
        elif fusion == "film":
            self.tab_enc = _mlp(tab_dim, tab_hidden)
            self.film = nn.Linear(tab_hidden, feat_dim * 2); fused = feat_dim
        elif fusion == "gate":
            self.tab_enc = _mlp(tab_dim, tab_hidden)
            self.gate = nn.Linear(tab_hidden, feat_dim); fused = feat_dim
        elif fusion == "attn":
            self.img_proj = nn.Linear(feat_dim, fuse_dim)
            self.tok_proj = nn.ModuleList([nn.Linear(1, fuse_dim) for _ in range(tab_dim)])
            self.attn = nn.TransformerEncoderLayer(
                d_model=fuse_dim, nhead=4, dim_feedforward=fuse_dim * 2,
                dropout=dropout, batch_first=True, activation="gelu")
            fused = fuse_dim
        else:
            raise ValueError(f"Unknown fusion '{fusion}'")
        self.fused_dim = fused
        self.reg_head = _make_reg_head(fused)

    def _hook(self, module, inp): self._feat = inp[0]

    def forward(self, img, tab):
        _ = self.image_model(img)            # backbone runs; hook captures the feature
        f = self._feat
        tab = tab.to(f.dtype)
        if self.fusion == "concat":
            fused = torch.cat([f, self.tab_enc(tab)], dim=-1)
        elif self.fusion == "film":
            g, b = self.film(self.tab_enc(tab)).chunk(2, dim=-1)
            fused = f * (1.0 + g) + b
        elif self.fusion == "gate":
            fused = f * torch.sigmoid(self.gate(self.tab_enc(tab)))
        else:
            toks = [self.img_proj(f).unsqueeze(1)]
            for i, proj in enumerate(self.tok_proj):
                toks.append(proj(tab[:, i:i+1]).unsqueeze(1))
            fused = self.attn(torch.cat(toks, dim=1))[:, 0]
        return self.reg_head(fused).squeeze(-1)

class ImageOnlyWrapper(nn.Module):
    """Image-only baseline: the captured feature goes to a fresh regression head of the
    same form as the fused model's head (LayerNorm, bottleneck, GELU, linear)."""
    def __init__(self, image_model):
        super().__init__()
        self.image_model = image_model
        self.head_name, head = _get_head(image_model)
        self.feat_dim = _infer_feat_dim(head)
        self._feat = None
        head.register_forward_pre_hook(self._hook)
        self.reg_head = _make_reg_head(self.feat_dim)
    def _hook(self, module, inp): self._feat = inp[0]
    def forward(self, img, tab):
        _ = self.image_model(img)
        return self.reg_head(self._feat).squeeze(-1)

print("Wrappers ready: MultiModalWrapper / ImageOnlyWrapper")


## Section 7 — Loss, metrics, early stopping

In [ ]:
def _norm_hb(hb):  return (hb - REG_CONFIG["hb_min"]) / (REG_CONFIG["hb_max"] - REG_CONFIG["hb_min"])
def _denorm_hb(x): return x * (REG_CONFIG["hb_max"] - REG_CONFIG["hb_min"]) + REG_CONFIG["hb_min"]

def build_reg_loss():
    fn = REG_CONFIG["loss_fn"].lower()
    if fn == "huber": return nn.HuberLoss(delta=REG_CONFIG["huber_delta"])
    if fn == "mae":   return nn.L1Loss()
    return nn.MSELoss()

def reg_metrics(hb_true, hb_pred):
    at, ap = np.asarray(hb_true, dtype=float), np.asarray(hb_pred, dtype=float)
    return {"mae":  float(np.mean(np.abs(at - ap))),
            "rmse": float(np.sqrt(np.mean((at - ap) ** 2))),
            "r2":   float(r2_score(at, ap)) if len(at) > 1 else float("nan")}

class EarlyStopper:
    def __init__(self, patience): self.p, self.n, self.best = patience, 0, float("inf")
    def step(self, s):
        if s < self.best - 1e-6: self.best, self.n = s, 0; return False
        self.n += 1; return self.p > 0 and self.n >= self.p

print("Regression loss and metrics ready.")

## Section 8 — Epoch loop

In [ ]:
def run_epoch(model, loader, reg_loss, optimizer=None, grad_accum=1, scaler=None):
    import contextlib
    training = optimizer is not None
    model.train() if training else model.eval()
    total, HBP, HBT, IDX = 0.0, [], [], []
    if training: optimizer.zero_grad()
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for bi, (imgs, tab, hb, idx) in enumerate(
                tqdm(loader, leave=False, desc="train" if training else "eval ")):
            imgs, tab, hb = imgs.to(DEVICE), tab.to(DEVICE), hb.to(DEVICE)
            amp = torch.cuda.amp.autocast() if scaler is not None else contextlib.nullcontext()
            with amp:
                hb_pred = model(imgs, tab)
                ht = _norm_hb(hb) if REG_CONFIG["normalize_hb"] else hb
                loss = reg_loss(hb_pred.view(-1), ht.view(-1))
            if training:
                (scaler.scale(loss / grad_accum) if scaler else (loss / grad_accum)).backward()
                if (bi + 1) % grad_accum == 0 or (bi + 1 == len(loader)):
                    if scaler:
                        scaler.unscale_(optimizer)
                        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        scaler.step(optimizer); scaler.update()
                    else:
                        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        optimizer.step()
                    optimizer.zero_grad()
            total += loss.item()
            hp = hb_pred.detach().view(-1)
            if REG_CONFIG["normalize_hb"]: hp = _denorm_hb(hp)
            HBP.extend(hp.cpu().tolist())
            HBT.extend(hb.cpu().view(-1).tolist())
            IDX.extend(idx.cpu().view(-1).tolist())
    m = {"loss": total / max(len(loader), 1)}
    m.update(reg_metrics(HBT, HBP))
    return m, HBT, HBP, IDX

print("run_epoch ready.")

## Section 9 — Cross-validation runner

`run_cv` trains K fold-models and always returns the **validation** metric. The locked
test set is scored only when `score_test=True`, which Stages 1–3 never set.

In [ ]:
FOLD_ROWS   = []   # one row per (config, fold)
SUMMARY_ROWS = []  # one row per config

def _fold_indices():
    skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)
    return list(skf.split(POOL_DF, POOL_LBL))

FOLDS = _fold_indices()
print(f"{K} folds prepared, sizes: {[len(v) for _, v in FOLDS]}")

def _build_loaders(tr_df, vl_df, numeric_feats, stats, want_test, shuffle_tab=False):
    T_TR, T_VA = build_transforms(True), build_transforms(False)
    kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
              pin_memory=(DEVICE == "cuda"), drop_last=False)
    st = dict(shuffle_tab=shuffle_tab)
    tr_ds = MMDataset(tr_df, T_TR, numeric_feats, stats, perm_seed=1, **st)
    vl_ld = DataLoader(MMDataset(vl_df, T_VA, numeric_feats, stats, perm_seed=2, **st),
                       shuffle=False, **kw)
    ts_ld = (DataLoader(MMDataset(TEST_DF, T_VA, numeric_feats, stats, perm_seed=3, **st),
                        shuffle=False, **kw) if want_test else None)
    if USE_AUGMENTATION:
        tr_ld = make_balanced_loader(dataset=tr_ds, hb_values=tr_df[HB_COL].values,
                                     batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                                     device=DEVICE, hb_filter_min=HB_FILTER_MIN,
                                     hb_filter_max=HB_FILTER_MAX, aug_bins=AUG_BINS)
    else:
        tr_ld = DataLoader(tr_ds, shuffle=True, **kw)
    return tr_ld, vl_ld, ts_ld

def _make_sched(opt):
    from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
    if WARMUP_EPOCHS > 0 and EPOCHS > WARMUP_EPOCHS:
        w = LinearLR(opt, start_factor=0.05, end_factor=1.0, total_iters=WARMUP_EPOCHS)
        c = CosineAnnealingLR(opt, T_max=max(EPOCHS - WARMUP_EPOCHS, 1))
        return SequentialLR(opt, [w, c], milestones=[WARMUP_EPOCHS])
    if SCHEDULER == "cosine": return CosineAnnealingLR(opt, T_max=EPOCHS)
    return None

_RESULTS_CACHE = f"{OUT_PREFIX}_all_configs.csv"

def _already_done(config_name):
    """Resume support: a long run can be interrupted and restarted without losing work."""
    if not RESUME or not os.path.exists(_RESULTS_CACHE): return None
    try:
        prev = pd.read_csv(_RESULTS_CACHE)
        hit = prev[prev.config == config_name]
        if len(hit): return hit.iloc[0].to_dict()
    except Exception:
        pass
    return None

def run_cv(config_name, model_factory, numeric_feats, stage="", score_test=False,
           collect_preds=False, seed=None, shuffle_tab=False, save_ckpt=False):
    """Train K fold-models. Returns a summary dict with validation metrics, plus test
    metrics only when score_test is True. The best-validation weights of each fold are
    restored before any scoring."""
    seed = SEED if seed is None else seed
    cached = _already_done(config_name)
    if cached is not None and (not score_test or "test_mae_mean" in cached):
        print(f"  [resume] {config_name}: reusing cached result "
              f"(val MAE {cached.get('val_mae_mean', float('nan')):.4f})")
        SUMMARY_ROWS.append(cached)
        return cached
    tab_dim = len(numeric_feats)
    sep = "=" * 74
    print(f"\n{sep}\n  {config_name}   [{stage}]   feats={numeric_feats or 'image-only'}"
          f"{'   + LOCKED TEST' if score_test else ''}\n{sep}")
    val_mae, val_rmse, val_r2 = [], [], []
    tst_mae, tst_rmse, tst_r2 = [], [], []
    pred_rows = []
    t0, params = time.time(), 0

    for fi, (tr_idx, vl_idx) in enumerate(FOLDS):
        tr_df, vl_df = POOL_DF.iloc[tr_idx], POOL_DF.iloc[vl_idx]
        seed_all(seed * 1000 + fi)
        stats = fit_tab_stats(tr_df, numeric_feats)
        tr_ld, vl_ld, ts_ld = _build_loaders(tr_df, vl_df, numeric_feats, stats, score_test,
                                             shuffle_tab=shuffle_tab)
        reg_loss = build_reg_loss()

        model = model_factory(tab_dim).to(DEVICE)
        if params == 0:
            params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        opt = torch.optim.AdamW(model.parameters(), lr=LR)
        sch = _make_sched(opt)
        scaler = torch.cuda.amp.GradScaler() if USE_AMP and DEVICE == "cuda" else None
        stop = EarlyStopper(EARLY_STOP)
        best_val, best_state = float("inf"), None

        for ep in range(1, EPOCHS + 1):
            tr, *_ = run_epoch(model, tr_ld, reg_loss, opt, GRAD_ACCUM, scaler)
            vl, *_ = run_epoch(model, vl_ld, reg_loss)
            if sch is not None: sch.step()
            if vl["loss"] < best_val:
                best_val = vl["loss"]
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(f"    F{fi+1} Ep{ep:02d}/{EPOCHS} TrL {tr['loss']:.4f} VlL {vl['loss']:.4f}"
                  f"  valMAE {vl['mae']:.3f} valRMSE {vl['rmse']:.3f}")
            if stop.step(vl["loss"]):
                print(f"    early stop @ ep{ep}"); break

        if best_state is not None: model.load_state_dict(best_state)
        if save_ckpt and best_state is not None:
            torch.save(best_state, f"{OUT_PREFIX}_ckpt_{_fn(config_name)}_f{fi+1}.pt")
        vl, *_ = run_epoch(model, vl_ld, reg_loss)
        val_mae.append(vl["mae"]); val_rmse.append(vl["rmse"]); val_r2.append(vl["r2"])
        row = dict(config=config_name, stage=stage, seed=seed, fold=fi + 1,
                   val_mae=vl["mae"], val_rmse=vl["rmse"], val_r2=vl["r2"])

        if score_test:
            ts, hbt, hbp, idx = run_epoch(model, ts_ld, reg_loss)
            tst_mae.append(ts["mae"]); tst_rmse.append(ts["rmse"]); tst_r2.append(ts["r2"])
            row.update(test_mae=ts["mae"], test_rmse=ts["rmse"], test_r2=ts["r2"])
            print(f"    F{fi+1} LOCKED TEST  MAE {ts['mae']:.4f}  RMSE {ts['rmse']:.4f}")
            if collect_preds:
                for i, t, p in zip(idx, hbt, hbp):
                    r = TEST_DF.iloc[int(i)]
                    pred_rows.append(dict(
                        config=config_name, seed=seed, fold=fi + 1,
                        patient_id=r.get(IMAGE_COL), hb_true=float(t), hb_pred=float(p),
                        age_months=float(r.get(AGE_COL, np.nan)),
                        sex=r.get(SEX_COL), who_threshold=float(r.get("_who_thr", np.nan))))
        FOLD_ROWS.append(row)
        del model
        if DEVICE == "cuda": torch.cuda.empty_cache()
        gc.collect()

    out = dict(config=config_name, stage=stage, seed=seed, params=params,
               time_s=time.time() - t0, folds=K, numeric_feats=list(numeric_feats),
               val_mae_mean=float(np.mean(val_mae)),  val_mae_std=float(np.std(val_mae, ddof=1)),
               val_rmse_mean=float(np.mean(val_rmse)),val_rmse_std=float(np.std(val_rmse, ddof=1)),
               val_r2_mean=float(np.mean(val_r2)))
    print("  " + "-" * 70)
    print(f"  {config_name}:  VAL MAE {out['val_mae_mean']:.4f} +/- {out['val_mae_std']:.4f}"
          f"   VAL RMSE {out['val_rmse_mean']:.4f}")
    if score_test:
        out.update(test_mae_mean=float(np.mean(tst_mae)), test_mae_std=float(np.std(tst_mae, ddof=1)),
                   test_rmse_mean=float(np.mean(tst_rmse)),test_rmse_std=float(np.std(tst_rmse, ddof=1)),
                   test_r2_mean=float(np.mean(tst_r2)))
        print(f"  {config_name}:  TEST MAE {out['test_mae_mean']:.4f} +/- {out['test_mae_std']:.4f}"
              f"   TEST RMSE {out['test_rmse_mean']:.4f}")
    SUMMARY_ROWS.append(out)
    # write after every configuration so an interrupted run loses at most one config
    pd.DataFrame(SUMMARY_ROWS).to_csv(_RESULTS_CACHE, index=False)
    pd.DataFrame(FOLD_ROWS).to_csv(f"{OUT_PREFIX}_per_fold.csv", index=False)
    if collect_preds and pred_rows:
        pd.DataFrame(pred_rows).to_csv(
            f"{OUT_PREFIX}_test_predictions_{_fn(config_name)}.csv", index=False)
        print(f"  per-sample test predictions saved ({len(pred_rows)} rows)")
    print(f"  done in {out['time_s']:.0f}s")
    return out

def _fail(name, e):
    print(f"  {name} FAILED: {e}"); traceback.print_exc()
    SUMMARY_ROWS.append(dict(config=name, status="FAIL", error=str(e)))

print("Nested-selection CV runner ready. Stages 1-3 never touch the test set.")

## Section 10 — Backbone registry (9 Mamba + 5 pretrained baselines)

In [ ]:
def _add(folder):
    p = os.path.join(MODELS_DIR, folder)
    if p not in sys.path: sys.path.insert(0, p)

def _load(mod_name, folder, filename):
    _add(folder)
    spec = importlib.util.spec_from_file_location(
        mod_name, os.path.join(MODELS_DIR, folder, filename))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
    return m

def _register():
    reg = {}
    try:
        m = _load("_m1", "01_Mamba_Official", "eye_hb_model.py")
        reg["Mamba1"] = lambda m=m: m.build_mamba1(IMG_SIZE, embed_dim=128, depth=4)
        reg["Mamba2"] = lambda m=m: m.build_mamba2(IMG_SIZE, embed_dim=128, depth=4)
    except Exception as e: print("  [skip] Mamba1/2:", e)
    try:
        m = _load("_m3", "06_Mamba3_Minimal", "eye_hb_model.py")
        reg["Mamba3"] = lambda m=m: m.build_model(IMG_SIZE, embed_dim=128, depth=4)
    except Exception as e: print("  [skip] Mamba3:", e)
    for key, folder in [("VMamba", "02_VMamba"), ("MambaVision", "03_MambaVision"),
                        ("MedMamba", "04_MedMamba")]:
        try:
            m = _load("_" + key, folder, "eye_hb_model.py")
            reg[key] = lambda m=m: m.build_model(IMG_SIZE)
        except Exception as e: print(f"  [skip] {key}:", e)
    try:
        m = _load("_em", "08_EfficientMamba", "eye_hb_model.py")
        reg["EfficientMamba"] = lambda m=m: m.build_model(
            img_size=IMG_SIZE, embed_dim=256, depth=4, freeze_backbone=FREEZE_BACKBONE)
    except Exception as e: print("  [skip] EfficientMamba:", e)
    try:
        m = _load("_asm", "09_AdaptiveScanMamba", "adaptive_scan_mamba.py")
        reg["AdaptiveScan"] = lambda m=m: m.build_adaptive_scan_mamba(
            img_size=IMG_SIZE, embed_dim=NEW_MODEL_CFG["embed_dim"],
            depth=NEW_MODEL_CFG["depth"], d_state=NEW_MODEL_CFG["d_state"],
            colour_proj=NEW_MODEL_CFG["colour_proj"])
    except Exception as e: print("  [skip] AdaptiveScan:", e)
    try:
        m = _load("_cf", "11_CrossFusionMamba", "cross_fusion_mamba.py")
        reg["CrossFusion"] = lambda m=m: m.build_cross_fusion_mamba(
            img_size=IMG_SIZE, embed_dim=NEW_MODEL_CFG["embed_dim"],
            depth=NEW_MODEL_CFG["depth"], d_state=NEW_MODEL_CFG["d_state"])
    except Exception as e: print("  [skip] CrossFusion:", e)
    try:
        import timm
        class TimmWrapper(nn.Module):
            """Exposes reg_head so the pre-hook has a terminal head to attach to.
            Inputs arrive scaled to [-1, 1]; they are mapped back to [0, 1] and then
            normalised with the statistics the pretrained weights expect."""
            def __init__(self, backbone_name, pretrained=True):
                super().__init__()
                self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                                                  num_classes=0)
                cfg = getattr(self.backbone, "pretrained_cfg", None) or \
                      getattr(self.backbone, "default_cfg", {}) or {}
                mean = cfg.get("mean", (0.485, 0.456, 0.406))
                std  = cfg.get("std",  (0.229, 0.224, 0.225))
                self.register_buffer("_mean", torch.tensor(mean).view(1, 3, 1, 1))
                self.register_buffer("_std",  torch.tensor(std).view(1, 3, 1, 1))
                if FREEZE_TIMM:
                    for p in self.backbone.parameters(): p.requires_grad = False
                fd = self.backbone.num_features
                self.reg_head = nn.Sequential(nn.LayerNorm(fd), nn.Linear(fd, 1))
            def forward(self, x):
                x = (x * 0.5 + 0.5 - self._mean) / self._std
                return self.reg_head(self.backbone(x)).squeeze(-1)
        for key, bb in [("ResNet50", "resnet50"), ("EfficientNetB3", "efficientnet_b3"),
                        ("DenseNet121", "densenet121"),
                        ("MobileNetV3", "mobilenetv3_large_100"),
                        ("ViT_B16", "vit_base_patch16_224")]:
            reg[key] = lambda bb=bb: TimmWrapper(bb, pretrained=True)
    except Exception as e: print("  [skip] timm baselines:", e)
    return reg

IMG_BUILDERS = _register()
_on = [k for k, v in RUN.items() if v]
print(f"Registered {len(IMG_BUILDERS)} builders; {len([k for k in _on if k in IMG_BUILDERS])}"
      f" of the {len(_on)} requested are available.")
missing = [k for k in _on if k not in IMG_BUILDERS]
if missing: print("  NOT available:", missing)

## Preflight — verify everything before the long run

Instantiates every requested backbone, pushes one dummy batch through the wrapper, and
checks paths, columns and images. A full run is many hours; this takes under a minute and
catches the failures that would otherwise surface an hour in.

In [ ]:
def preflight(n_probe=2):
    ok = True
    print("=" * 74)

    # ---- 1. data ----
    print("DATA")
    print(f"  rows after Hb filter [{HB_FILTER_MIN}, {HB_FILTER_MAX}] : {len(FULL)}")
    print(f"  pool {len(POOL_DF)}  |  locked test {len(TEST_DF)}  |  folds {K}")
    for c in NUMERIC_FEATURES:
        n = pd.to_numeric(FULL[c], errors="coerce").notna().sum()
        flag = "ok " if n > 0.9 * len(FULL) else "LOW"
        print(f"  [{flag}] {c:>8}: {n}/{len(FULL)} usable")
        if n == 0: ok = False
    n_age = FULL[AGE_COL].notna().sum()
    print(f"  [{'ok ' if n_age else 'WARN'}] age    : {n_age}/{len(FULL)} usable")

    # ---- 2. images readable ----
    print("\nIMAGES")
    missing = 0
    for _, r in FULL.head(50).iterrows():
        try:
            Image.open(r["_img_path"]).convert("RGB")
        except Exception:
            missing += 1
    print(f"  [{'ok ' if missing == 0 else 'FAIL'}] {50 - missing}/50 sample images open cleanly")
    if missing: ok = False

    # ---- 3. every requested backbone builds, exposes a head, and runs ----
    print("\nBACKBONES (build + forward through the wrapper)")
    dummy_img = torch.randn(2, 3, IMG_SIZE, IMG_SIZE)
    dummy_tab = torch.randn(2, len(NUMERIC_FEATURES))
    for name, on in RUN.items():
        if not on: continue
        if name not in IMG_BUILDERS:
            print(f"  [FAIL] {name:<16} no builder registered"); ok = False; continue
        try:
            m = MultiModalWrapper(IMG_BUILDERS[name](), len(NUMERIC_FEATURES), fusion=FUSION)
            m.eval()
            with torch.no_grad():
                out = m(dummy_img, dummy_tab)
            n_par = sum(p.numel() for p in m.parameters() if p.requires_grad)
            assert out.shape == (2,), f"expected (2,), got {tuple(out.shape)}"
            print(f"  [ok ] {name:<16} head={m.head_name:<9} feat={m.feat_dim:<5} "
                  f"params={n_par/1e6:5.1f}M")
            del m
        except Exception as e:
            print(f"  [FAIL] {name:<16} {type(e).__name__}: {e}"); ok = False
        gc.collect()

    # ---- 4. image-only wrapper ----
    try:
        m = ImageOnlyWrapper(IMG_BUILDERS[[k for k, v in RUN.items() if v and k in IMG_BUILDERS][0]]())
        m.eval()
        with torch.no_grad(): m(dummy_img, torch.zeros(2, 0))
        print("  [ok ] ImageOnlyWrapper forward")
        del m
    except Exception as e:
        print(f"  [FAIL] ImageOnlyWrapper: {e}"); ok = False

    # ---- 5. size of the job ----
    n_cfg = (len([k for k, v in RUN.items() if v and k in IMG_BUILDERS]) * int(RUN_STAGE1)
             + len(FUSION_MODES) * int(RUN_STAGE2)
             + len(VARIABLE_STEPS) * int(RUN_STAGE3)
             + (2 * len(FINAL_SEEDS) + 2 * int(RUN_CONTROLS)) * int(RUN_FINAL))
    print(f"\nJOB SIZE")
    print(f"  {n_cfg} configurations x {K} folds = {n_cfg * K} training runs, "
          f"up to {EPOCHS} epochs each")
    print(f"  profile={RUN_PROFILE}  resume={RESUME}")
    print("=" * 74)
    print("PREFLIGHT PASSED -- safe to run the stages" if ok else
          "PREFLIGHT FAILED -- fix the items marked FAIL before running")
    return ok

PREFLIGHT_OK = preflight()

## Stage 1 — Backbone selection (validation only)

The fourteen backbones are trained on the image alone (`STAGE1_FEATS = []`), which is
what the paper states. Ranking uses validation MAE. **The test set is not read here.**


In [ ]:
def make_factory(img_build_fn, numeric_feats, fusion):
    if len(numeric_feats) == 0:
        return lambda tab_dim: ImageOnlyWrapper(img_build_fn())
    return lambda tab_dim: MultiModalWrapper(img_build_fn(), tab_dim, fusion=fusion)

STAGE1 = []
if RUN_STAGE1:
    for name, on in RUN.items():
        if not on: continue
        if name not in IMG_BUILDERS:
            print(f"{name}: no builder, skipped."); continue
        try:
            f = make_factory(IMG_BUILDERS[name], STAGE1_FEATS, FUSION)
            STAGE1.append(run_cv(name, f, STAGE1_FEATS,
                                 stage="S1-backbone", score_test=False))
        except Exception as e:
            _fail(name, e)

    s1 = pd.DataFrame([r for r in STAGE1 if "val_mae_mean" in r]).sort_values("val_mae_mean")
    print("\n=== Stage 1 ranking by VALIDATION MAE ===")
    print(s1[["config", "val_mae_mean", "val_mae_std", "val_rmse_mean", "params"]].to_string(index=False))
    s1.to_csv(f"{OUT_PREFIX}_stage1_backbones.csv", index=False)
    BEST_BACKBONE = s1.iloc[0]["config"]
    print(f"\nSelected backbone (validation): {BEST_BACKBONE}")
else:
    BEST_BACKBONE = "ViT_B16"
    print("Stage 1 skipped; BEST_BACKBONE =", BEST_BACKBONE)

## Stage 2 — Fusion selection (validation only)

The backbone chosen in Stage 1 is fixed; the four fusion operators are compared on the
same folds, again on validation.

In [ ]:
STAGE2 = []
if RUN_STAGE2:
    for fmode in FUSION_MODES:
        try:
            f = make_factory(IMG_BUILDERS[BEST_BACKBONE], NUMERIC_FEATURES, fmode)
            STAGE2.append(run_cv(f"{BEST_BACKBONE}/{fmode}", f, NUMERIC_FEATURES,
                                 stage="S2-fusion", score_test=False))
        except Exception as e:
            _fail(f"{BEST_BACKBONE}/{fmode}", e)

    s2 = pd.DataFrame([r for r in STAGE2 if "val_mae_mean" in r]).sort_values("val_mae_mean")
    print("\n=== Stage 2 ranking by VALIDATION MAE ===")
    print(s2[["config", "val_mae_mean", "val_mae_std", "val_rmse_mean"]].to_string(index=False))
    s2.to_csv(f"{OUT_PREFIX}_stage2_fusion.csv", index=False)
    BEST_FUSION = s2.iloc[0]["config"].split("/")[-1]
    print(f"\nSelected fusion (validation): {BEST_FUSION}")
else:
    BEST_FUSION = FUSION
    print("Stage 2 skipped; BEST_FUSION =", BEST_FUSION)

## Stage 3 — Clinical variable selection (validation only)

Backbone and fusion are fixed. Each variable is added on its own, then all three
together. Still validation only.

In [ ]:
STAGE3 = []
if RUN_STAGE3:
    for label, feats in VARIABLE_STEPS:
        try:
            f = make_factory(IMG_BUILDERS[BEST_BACKBONE], feats, BEST_FUSION)
            STAGE3.append(run_cv(f"{BEST_BACKBONE}/{BEST_FUSION}/{label}", f, feats,
                                 stage="S3-variables", score_test=False))
        except Exception as e:
            _fail(f"{BEST_BACKBONE}/{BEST_FUSION}/{label}", e)

    s3 = pd.DataFrame([r for r in STAGE3 if "val_mae_mean" in r]).sort_values("val_mae_mean")
    print("\n=== Stage 3 ranking by VALIDATION MAE ===")
    print(s3[["config", "val_mae_mean", "val_mae_std", "val_rmse_mean"]].to_string(index=False))
    s3.to_csv(f"{OUT_PREFIX}_stage3_variables.csv", index=False)
    _bf = s3.iloc[0]["numeric_feats"]          # a string when read back from the resume cache
    BEST_FEATS = list(json.loads(_bf.replace("'", '"')) if isinstance(_bf, str) else _bf)
    print(f"\nSelected variables (validation): {BEST_FEATS or 'image-only'}")
else:
    BEST_FEATS = NUMERIC_FEATURES
    print("Stage 3 skipped; BEST_FEATS =", BEST_FEATS)

## Final — the pre-specified evaluations on the locked test set

The list below is fixed before the test set is read and is printed before any training:

1. **MultiHb**, the configuration selected on validation, once per seed.
2. **ImageOnly**, the same backbone without the clinical variables, once per seed.
3. **ShuffledTab**, the same multimodal model trained and tested with the tabular vector
   of a different subject. It has MultiHb's capacity and none of its information, so the
   gap between it and MultiHb is what the variables themselves contribute.
4. **MultiHb+AgeSex**, the image with height, weight, BMI, age and sex.

Per-subject predictions are written to CSV for every one of them.


In [ ]:

FINAL, PLAN = [], []
if RUN_FINAL:
    MM_FEATS = list(BEST_FEATS) if BEST_FEATS else list(NUMERIC_FEATURES)
    if not BEST_FEATS:
        print("[note] Stage 3 selected image-only on validation. Report that. The full "
              "variable set is still scored below so the comparison can be shown.")
    bb = IMG_BUILDERS[BEST_BACKBONE]
    for sd in FINAL_SEEDS:
        PLAN.append((f"MultiHb_s{sd}", make_factory(bb, MM_FEATS, BEST_FUSION), MM_FEATS,
                     dict(seed=sd, save_ckpt=(SAVE_CHECKPOINTS and sd == FINAL_SEEDS[0]))))
        PLAN.append((f"ImageOnly_s{sd}", make_factory(bb, [], BEST_FUSION), [], dict(seed=sd)))
    if RUN_CONTROLS:
        sd = FINAL_SEEDS[0]
        ALL_FEATS = MM_FEATS + [AGE_COL, SEX_NUM_COL]
        PLAN.append((f"ShuffledTab_s{sd}", make_factory(bb, MM_FEATS, BEST_FUSION), MM_FEATS,
                     dict(seed=sd, shuffle_tab=True)))
        PLAN.append((f"MultiHb+AgeSex_s{sd}", make_factory(bb, ALL_FEATS, BEST_FUSION), ALL_FEATS,
                     dict(seed=sd)))

    print("PRE-SPECIFIED TEST EVALUATIONS")
    print(f"  backbone {BEST_BACKBONE} | fusion {BEST_FUSION} | variables {MM_FEATS}")
    for name, _, feats, kw in PLAN:
        print(f"  - {name:<24} feats={feats or 'image-only'}")
    json.dump(dict(backbone=BEST_BACKBONE, fusion=BEST_FUSION, feats=MM_FEATS,
                   seeds=FINAL_SEEDS, plan=[p[0] for p in PLAN]),
              open(f"{OUT_PREFIX}_final_plan.json", "w"), indent=1)

    for name, factory, feats, kw in PLAN:
        try:
            FINAL.append(run_cv(name, factory, feats, stage="FINAL",
                                score_test=True, collect_preds=True, **kw))
        except Exception as e:
            _fail(name, e)

    fin = pd.DataFrame([r for r in FINAL if "test_mae_mean" in r])
    if len(fin):
        print("\n=== LOCKED TEST RESULTS, mean over folds (SD is the sample SD across folds) ===")
        print(fin[["config", "val_mae_mean", "test_mae_mean", "test_mae_std",
                   "test_rmse_mean", "test_r2_mean", "params"]].to_string(index=False))
        fin.to_csv(f"{OUT_PREFIX}_final_test.csv", index=False)

pd.DataFrame(FOLD_ROWS).to_csv(f"{OUT_PREFIX}_per_fold.csv", index=False)
pd.DataFrame(SUMMARY_ROWS).to_csv(f"{OUT_PREFIX}_all_configs.csv", index=False)
print(f"\nSaved per-fold metrics ({len(FOLD_ROWS)} rows) and config summaries "
      f"({len(SUMMARY_ROWS)} rows).")


## Analysis — baselines, paired tests, agreement, stratified error

Everything below reads the saved per-subject predictions. Nothing is retrained, so this
cell can be re-run freely.

The unit of analysis is the **subject**. Each subject's prediction is the mean over the
fold models and seeds of a configuration. Uncertainty comes from a paired bootstrap over
the test subjects, which is the uncertainty a reader cares about: how the error would
change with a different sample of children.


In [ ]:

def _metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    e = p - y
    ss_res, ss_tot = float((e ** 2).sum()), float(((y - y.mean()) ** 2).sum())
    return dict(mae=float(np.abs(e).mean()), rmse=float(np.sqrt((e ** 2).mean())),
                r2=1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan"),
                bias=float(e.mean()), sd=float(e.std(ddof=1)),
                r=float(np.corrcoef(y, p)[0, 1]) if len(y) > 2 else float("nan"))

MODEL_MAE = {}     # per fold-model test MAE, keyed by configuration
def subject_preds(base):
    """Fold- and seed-averaged prediction per subject for configs named '<base>_s<seed>'."""
    files = sorted(glob.glob(f"{OUT_PREFIX}_test_predictions_{_fn(base)}_s*.csv"))
    if not files: return None
    d = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    g = d.groupby("patient_id").agg(hb_true=("hb_true", "first"), hb_pred=("hb_pred", "mean"),
                                    age_months=("age_months", "first"), sex=("sex", "first"),
                                    who_threshold=("who_threshold", "first")).reset_index()
    MODEL_MAE[base] = np.asarray([float(np.abs(x.hb_pred - x.hb_true).mean())
                                  for _, x in d.groupby(["seed", "fold"])])
    return g

def boot_ci(values, n_boot=None, seed=0):
    """Percentile bootstrap 95% CI for the mean of per-subject values."""
    n_boot = N_BOOT if n_boot is None else n_boot
    v = np.asarray(values, float); rng = np.random.RandomState(seed)
    idx = rng.randint(0, len(v), size=(n_boot, len(v)))
    m = v[idx].mean(axis=1)
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5)), m

def paired_compare(ga, gb, label):
    """Paired comparison of absolute error per subject. Positive difference = b is better."""
    m = ga.merge(gb, on="patient_id", suffixes=("_a", "_b"))
    d = np.abs(m.hb_pred_a - m.hb_true_a).values - np.abs(m.hb_pred_b - m.hb_true_b).values
    lo, hi, bm = boot_ci(d)
    p_boot = float(min(1.0, max(2 * min((bm <= 0).mean(), (bm >= 0).mean()), 1.0 / len(bm))))  # floor: 1/resamples
    try:
        from scipy import stats as sps
        p_w = float(sps.wilcoxon(d).pvalue)
    except Exception:
        p_w = float("nan")
    print(f"  {label}")
    print(f"     mean difference in absolute error {d.mean():+.4f} g/dL, "
          f"95% CI [{lo:+.4f}, {hi:+.4f}], n = {len(d)} subjects")
    print(f"     paired bootstrap p = {p_boot:.4f} (two-sided, {N_BOOT} resamples); "
          f"Wilcoxon signed-rank p = {p_w:.4f}")
    return dict(comparison=label, n=len(d), diff=float(d.mean()), ci_lo=lo, ci_hi=hi,
                p_boot=p_boot, p_wilcoxon=p_w)

def holm(pvals):
    p = np.asarray(pvals, float); order = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (len(p) - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

ROWS = []
def _report(name, y, p, note=""):
    m = _metrics(y, p)
    lo, hi, _ = boot_ci(np.abs(np.asarray(p, float) - np.asarray(y, float)))
    ROWS.append(dict(model=name, n=len(y), mae=m["mae"], mae_ci_lo=lo, mae_ci_hi=hi,
                     rmse=m["rmse"], r2=m["r2"], bias=m["bias"],
                     loa_lo=m["bias"] - 1.96 * m["sd"], loa_hi=m["bias"] + 1.96 * m["sd"],
                     pearson_r=m["r"], note=note))

# ---------- 1. reference predictors that use no image ----------
y_pool, y_test = POOL_DF[HB_COL].values.astype(float), TEST_DF[HB_COL].values.astype(float)
if RUN_BASELINES:
    _report("Predict pool mean",   y_test, np.full_like(y_test, y_pool.mean()), "no model")
    _report("Predict pool median", y_test, np.full_like(y_test, np.median(y_pool)), "no model")

    def _design(df, feats, ref):
        X = np.column_stack([pd.to_numeric(df[c], errors="coerce").values for c in feats]).astype(float)
        R = np.column_stack([pd.to_numeric(ref[c], errors="coerce").values for c in feats]).astype(float)
        mu, sd = np.nanmean(R, 0), np.nanstd(R, 0) + 1e-9
        X = (X - mu) / sd
        X[np.isnan(X)] = 0.0                       # missing -> pool mean
        return X
    def _ridge(feats, lam=1.0):
        Xtr, Xte = _design(POOL_DF, feats, POOL_DF), _design(TEST_DF, feats, POOL_DF)
        Xtr = np.column_stack([Xtr, Xtr ** 2, np.ones(len(Xtr))])     # quadratic terms
        Xte = np.column_stack([Xte, Xte ** 2, np.ones(len(Xte))])
        P = lam * np.eye(Xtr.shape[1]); P[-1, -1] = 0.0
        w = np.linalg.solve(Xtr.T @ Xtr + P, Xtr.T @ y_pool)
        return Xte @ w
    TAB_SETS = [("height, weight, BMI", NUMERIC_FEATURES),
                ("age, sex", [AGE_COL, SEX_NUM_COL]),
                ("height, weight, BMI, age, sex", NUMERIC_FEATURES + [AGE_COL, SEX_NUM_COL])]
    for lab, feats in TAB_SETS:
        _report(f"Tabular only, ridge: {lab}", y_test, _ridge(feats), "no image")
        try:
            from sklearn.ensemble import GradientBoostingRegressor
            gb = GradientBoostingRegressor(random_state=SEED).fit(_design(POOL_DF, feats, POOL_DF), y_pool)
            _report(f"Tabular only, boosting: {lab}", y_test,
                    gb.predict(_design(TEST_DF, feats, POOL_DF)), "no image")
        except Exception as e:
            print("  [skip] boosting baseline:", e)

# ---------- 2. the image models ----------
G = {b: subject_preds(b) for b in ("ImageOnly", "MultiHb", "ShuffledTab", "MultiHb+AgeSex")}
for b, g in G.items():
    if g is not None:
        pm = MODEL_MAE[b]
        _report(b, g.hb_true, g.hb_pred,
                f"mean of {len(pm)} models; single-model MAE "
                f"{pm.mean():.4f} +/- {pm.std(ddof=1) if len(pm) > 1 else 0:.4f} (sample SD)")

if ROWS:
    T = pd.DataFrame(ROWS)
    pd.set_option("display.width", 220)
    print("TEST-SET RESULTS, one row per model (MAE with 95% bootstrap CI over subjects)")
    print(T[["model", "n", "mae", "mae_ci_lo", "mae_ci_hi", "rmse", "r2", "bias",
             "loa_lo", "loa_hi", "pearson_r"]].round(3).to_string(index=False))
    for r in ROWS:
        if r["note"].startswith("mean of"): print(f"  {r['model']}: {r['note']}")
    T.to_csv(f"{OUT_PREFIX}_results_table.csv", index=False)

# ---------- 3. paired comparisons ----------
print("\nPAIRED COMPARISONS (positive difference means the second model has lower error)")
CMP = []
if G["ImageOnly"] is not None and G["MultiHb"] is not None:
    print(" Primary, pre-specified:")
    CMP.append(paired_compare(G["ImageOnly"], G["MultiHb"], "ImageOnly  ->  MultiHb"))
sec = []
if G["ShuffledTab"] is not None and G["MultiHb"] is not None:
    sec.append(paired_compare(G["ShuffledTab"], G["MultiHb"], "ShuffledTab  ->  MultiHb  (capacity control)"))
if G["MultiHb"] is not None and G["MultiHb+AgeSex"] is not None:
    sec.append(paired_compare(G["MultiHb"], G["MultiHb+AgeSex"], "MultiHb  ->  MultiHb+AgeSex"))
if G["ImageOnly"] is not None and G["ShuffledTab"] is not None:
    sec.append(paired_compare(G["ImageOnly"], G["ShuffledTab"], "ImageOnly  ->  ShuffledTab  (capacity alone)"))
if sec:
    adj = holm([c["p_boot"] for c in sec])
    for c, a in zip(sec, adj):
        c["p_boot_holm"] = float(a)
        print(f"     Holm-adjusted p, {c['comparison']}: {a:.4f}")
    CMP += sec
if CMP: pd.DataFrame(CMP).to_csv(f"{OUT_PREFIX}_paired_comparisons.csv", index=False)

# ---------- 4. stratified error and figures for the selected model ----------
g = G["MultiHb"]
if g is not None:
    err = (g.hb_pred - g.hb_true)
    print("\nMultiHb, ERROR BY Hb BAND (bias > 0 means over-estimation)")
    band_rows = []
    for lo, hi, lab in [(0, 7, "<7"), (7, 9, "7-9"), (9, 11, "9-11"), (11, 13, "11-13"), (13, 99, ">=13")]:
        m = (g.hb_true >= lo) & (g.hb_true < hi)
        if m.sum():
            band_rows.append(dict(band=f"Hb {lab}", n=int(m.sum()),
                                  mae=float(err[m].abs().mean()), bias=float(err[m].mean())))
    for lo, hi, lab in [(0, 24, "<2 y"), (24, 60, "2-5 y"), (60, 120, "5-10 y"), (120, 999, ">=10 y")]:
        m = (g.age_months >= lo) & (g.age_months < hi)
        if m.sum():
            band_rows.append(dict(band=f"age {lab}", n=int(m.sum()),
                                  mae=float(err[m].abs().mean()), bias=float(err[m].mean())))
    for s_ in g.sex.dropna().unique():
        m = g.sex == s_
        band_rows.append(dict(band=f"sex {s_}", n=int(m.sum()),
                              mae=float(err[m].abs().mean()), bias=float(err[m].mean())))
    B = pd.DataFrame(band_rows); print(B.round(3).to_string(index=False))
    B.to_csv(f"{OUT_PREFIX}_stratified_error.csv", index=False)
    g.to_csv(f"{OUT_PREFIX}_subject_level_predictions.csv", index=False)

    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    mt = _metrics(g.hb_true, g.hb_pred)
    fig, ax = plt.subplots(1, 2, figsize=(9, 4.2), dpi=300)
    lim = [min(g.hb_true.min(), g.hb_pred.min()) - 0.5, max(g.hb_true.max(), g.hb_pred.max()) + 0.5]
    ax[0].scatter(g.hb_true, g.hb_pred, s=8, alpha=0.55, linewidths=0)
    ax[0].plot(lim, lim, "k--", lw=0.8); ax[0].set_xlim(lim); ax[0].set_ylim(lim)
    ax[0].set_xlabel("Laboratory Hb (g/dL)"); ax[0].set_ylabel("Estimated Hb (g/dL)")
    ax[0].set_title(f"MAE {mt['mae']:.2f} g/dL, R2 {mt['r2']:.2f}, n = {len(g)}", fontsize=9)
    mean_ = (g.hb_true + g.hb_pred) / 2
    ax[1].scatter(mean_, err, s=8, alpha=0.55, linewidths=0)
    for yv, ls in [(mt["bias"], "-"), (mt["bias"] - 1.96 * mt["sd"], "--"), (mt["bias"] + 1.96 * mt["sd"], "--")]:
        ax[1].axhline(yv, color="k", ls=ls, lw=0.8)
    ax[1].set_xlabel("Mean of estimate and laboratory Hb (g/dL)")
    ax[1].set_ylabel("Estimate minus laboratory (g/dL)")
    ax[1].set_title(f"bias {mt['bias']:+.2f}, limits of agreement "
                    f"[{mt['bias'] - 1.96 * mt['sd']:+.2f}, {mt['bias'] + 1.96 * mt['sd']:+.2f}]", fontsize=9)
    for a in ax:
        a.spines["top"].set_visible(False); a.spines["right"].set_visible(False)
    fig.tight_layout(); fig.savefig(f"{OUT_PREFIX}_agreement.png", facecolor="white"); plt.close(fig)
    print(f"\nSaved {OUT_PREFIX}_agreement.png (scatter and Bland-Altman).")
else:
    print("\nNo MultiHb predictions found; run the Final cell first.")


## Out-of-range evaluation

Scores the saved MultiHb fold models on subjects whose laboratory Hb lies outside the
inclusion range. These subjects were never used for training, validation or testing. The
result says what the estimator returns for severe anemia and for very high values, which
the paper must state alongside the working range.

In [ ]:
def predict_df(df, base_name, feats):
    """Mean prediction of the saved fold models of `base_name` on the rows of df."""
    preds = []
    for fi, (tr_idx, _) in enumerate(FOLDS):
        ck = f"{OUT_PREFIX}_ckpt_{_fn(base_name)}_f{fi+1}.pt"
        if not os.path.exists(ck): continue
        stats = fit_tab_stats(POOL_DF.iloc[tr_idx], feats)
        ld = DataLoader(MMDataset(df, build_transforms(False), feats, stats),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
        model = make_factory(IMG_BUILDERS[BEST_BACKBONE], feats, BEST_FUSION)(len(feats))
        model.load_state_dict(torch.load(ck, map_location="cpu")); model.to(DEVICE).eval()
        out = []
        with torch.no_grad():
            for imgs, tab, hb, idx in ld:
                p = model(imgs.to(DEVICE), tab.to(DEVICE)).view(-1)
                if REG_CONFIG["normalize_hb"]: p = _denorm_hb(p)
                out.extend(p.cpu().tolist())
        preds.append(out); del model
        if DEVICE == "cuda": torch.cuda.empty_cache()
    return np.mean(np.asarray(preds), axis=0) if preds else None

if RUN_OUT_OF_RANGE and RUN_FINAL and len(OUT_DF):
    base = f"MultiHb_s{FINAL_SEEDS[0]}"
    p = predict_df(OUT_DF, base, MM_FEATS)
    if p is None:
        print("No checkpoints found. Set SAVE_CHECKPOINTS = True and run the Final cell.")
    else:
        o = OUT_DF[[IMAGE_COL, HB_COL, AGE_COL]].copy(); o["hb_pred"] = p
        o["error"] = o.hb_pred - o[HB_COL]
        print(f"OUT-OF-RANGE SUBJECTS, inclusion range [{HB_FILTER_MIN}, {HB_FILTER_MAX}] g/dL")
        for lab, m in [(f"Hb < {HB_FILTER_MIN}", o[HB_COL] < HB_FILTER_MIN),
                       (f"Hb > {HB_FILTER_MAX}", o[HB_COL] > HB_FILTER_MAX)]:
            if m.sum():
                print(f"  {lab:>10}: n = {int(m.sum()):3d}   lab mean {o[HB_COL][m].mean():.2f}   "
                      f"estimate mean {o.hb_pred[m].mean():.2f}   MAE {o.error[m].abs().mean():.2f}   "
                      f"bias {o.error[m].mean():+.2f}")
        o.to_csv(f"{OUT_PREFIX}_out_of_range_predictions.csv", index=False)
else:
    print("Out-of-range evaluation skipped "
          f"(rows outside the range: {len(OUT_DF)}; RUN_OUT_OF_RANGE = {RUN_OUT_OF_RANGE}).")

## Device benchmark

Run this cell on the handheld with `RUN_DEVICE_BENCHMARK = True`, after copying the
repository, the checkpoint `*_ckpt_MultiHb_s<seed>_f1.pt`, the prediction CSV and the test
images. Run Sections 1 to 10 first, with the stage switches off. It reports latency per
image on the CPU, peak memory, and how far the device's output moves from the output
saved on the training machine. Report the hardware model alongside these numbers.

In [ ]:
if RUN_DEVICE_BENCHMARK:
    import resource, platform
    base = f"MultiHb_s{FINAL_SEEDS[0]}"
    plan = json.load(open(f"{OUT_PREFIX}_final_plan.json"))
    feats, ck = plan["feats"], f"{OUT_PREFIX}_ckpt_{_fn(base)}_f1.pt"
    model = make_factory(IMG_BUILDERS[plan["backbone"]], feats, plan["fusion"])(len(feats))
    model.load_state_dict(torch.load(ck, map_location="cpu")); model.eval()
    stats = fit_tab_stats(POOL_DF.iloc[FOLDS[0][0]], feats)
    ds = MMDataset(TEST_DF, build_transforms(False), feats, stats)
    n = min(len(ds), 100); lat, out = [], []
    with torch.no_grad():
        for i in range(n):
            img, tab, hb, _ = ds[i]
            t0 = time.perf_counter()
            p = model(img.unsqueeze(0), tab.unsqueeze(0)).view(-1)
            lat.append((time.perf_counter() - t0) * 1000)
            out.append(float(_denorm_hb(p)[0]) if REG_CONFIG["normalize_hb"] else float(p[0]))
    lat = np.asarray(lat[5:])                      # drop warm-up
    ref = pd.read_csv(f"{OUT_PREFIX}_test_predictions_{_fn(base)}.csv")
    ref = ref[ref.fold == 1]
    ref = pd.Series(ref.hb_pred.values, index=ref.patient_id.astype(str).values)
    ids = TEST_DF[IMAGE_COL].iloc[:n].astype(str).values
    d = np.asarray([out[i] - ref.get(ids[i], np.nan) for i in range(n)], float)
    print("DEVICE BENCHMARK")
    print(f"  machine            : {platform.machine()} | {platform.processor() or platform.platform()}")
    print(f"  torch threads      : {torch.get_num_threads()}")
    print(f"  latency per image  : median {np.median(lat):.0f} ms, 95th percentile {np.percentile(lat, 95):.0f} ms (n = {len(lat)})")
    print(f"  peak memory (RSS)  : {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024:.0f} MB")
    print(f"  parameters         : {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M")
    print(f"  agreement with the saved predictions: mean |difference| {np.nanmean(np.abs(d)):.4f} g/dL, "
          f"max {np.nanmax(np.abs(d)):.4f} g/dL")
else:
    print("Device benchmark skipped (RUN_DEVICE_BENCHMARK = False).")

## Outputs

| File | Contents |
|---|---|
| `multihb_exp_stage1_backbones.csv` | validation metrics, 14 backbones |
| `multihb_exp_stage2_fusion.csv` | validation metrics, 4 fusion operators |
| `multihb_exp_stage3_variables.csv` | validation metrics, variable subsets |
| `multihb_exp_final_plan.json` | the pre-specified test evaluations, written before training |
| `multihb_exp_final_test.csv` | locked-test metrics per configuration and seed |
| `multihb_exp_per_fold.csv` | every fold of every configuration |
| `multihb_exp_test_predictions_*.csv` | per-subject test predictions, per fold and seed |
| `multihb_exp_results_table.csv` | one row per model: MAE with 95% CI, RMSE, R2, bias, limits of agreement |
| `multihb_exp_paired_comparisons.csv` | paired bootstrap and Wilcoxon results |
| `multihb_exp_stratified_error.csv` | MultiHb error by Hb band, age band and sex |
| `multihb_exp_agreement.png` | scatter and Bland-Altman plot |
| `multihb_exp_out_of_range_predictions.csv` | estimates for subjects outside the inclusion range |
| `multihb_exp_ckpt_*.pt` | fold models of the selected configuration |
